Document Summarization using Iterative Refinement.

Use case -
1. Summarize large files 
2. Summarize Research Papers

In [7]:
# !pip3 install sentence-transformers

In [8]:
from langchain_mistralai.chat_models import ChatMistralAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter, SentenceTransformersTokenTextSplitter
from langchain_core.documents import Document
from langchain_core.runnables import RunnableConfig
from langgraph.graph import START, END, StateGraph
from typing import TypedDict, List


Load Document for Summarization 

In [9]:
def load_doc(file_path : str):
    document_loader = PyPDFLoader(file_path)
    documents = document_loader.load()
    # text_splitter = SentenceTransformersTokenTextSplitter(tokens_per_chunk=200, chunk_overlap=50)
    # splitted_doc = text_splitter.split_documents(documents=documents)
    return documents

In [10]:
documents = load_doc("transformers.pdf") ##"STM_8T.pdf"
for doc in documents:
    print(doc.page_content)

An Introduction to Transformers
Richard E. Turner
Department of Engineering, University of Cambridge, UK
Microsoft Research, Cambridge, UK
ret26@cam.ac.uk
Abstract. The transformer is a neural network component that can be used to learn useful represen-
tations of sequences or sets of data-points [Vaswani et al., 2017]. The transformer has driven recent
advances in natural language processing [Devlin et al., 2019], computer vision [Dosovitskiy et al., 2021],
and spatio-temporal modelling [Bi et al., 2022]. There are many introductions to transformers, but most
do not contain precise mathematical descriptions of the architecture and the intuitions behind the design
choices are often also missing.1 Moreover, as research takes a winding path, the explanations for the
components of the transformer can be idiosyncratic. In this note we aim for a mathematically precise,
intuitive, and clean description of the transformer architecture. We will not discuss training as this is
rather standard. 

The LLM

In [11]:
api_key = "Eajkd7toYyYCEoU1LQiNFcPTvyK3ONep"
llm = ChatMistralAI(api_key=api_key, model_name= "mistral-large-latest")

The Nodes : generate, refine, END

In [12]:
class State(TypedDict):
    content : List[str]
    summary : str
    index : int

In [13]:
async def generate_initial_summary(state : State):
    prompt = ChatPromptTemplate.from_template(""" Write a concise summary for the given content.
                                            {content}""")
    initial_summary_chain = prompt | llm
    initial_summary = await initial_summary_chain.ainvoke(input={"content":state["content"][0]})
    return {"summary": initial_summary, "index":1}


In [14]:
async def generate_refined_summary(state : State):
    refine_prompt = ChatPromptTemplate.from_template(""" With the given content and summary, refine the summary.
                                                    {content}

                                                    {summary}
                                                    """)
    refinement_chain = refine_prompt | llm 
    refined_summary = await refinement_chain.ainvoke(input={"summary" : state["summary"], "content" : state["content"][state["index"]]})

    return {"summary" : refined_summary, "index": state["index"]+1}


The Router

In [15]:
def route(state:State):
    if state["index"] >= len(state["content"]):
        return END 
    else :
       return "generate_refined_summary"

In [16]:
graph_builder = StateGraph(State)
graph_builder.add_node("generate_initial_summary",generate_initial_summary)
graph_builder.add_node("generate_refined_summary", generate_refined_summary)
graph_builder.add_edge(START, "generate_initial_summary")
graph_builder.add_conditional_edges("generate_initial_summary", route)
graph_builder.add_conditional_edges("generate_refined_summary", route)

In [17]:
graph = graph_builder.compile()
graph.get_graph()

Graph(nodes={'__start__': Node(id='__start__', name='__start__', data=RunnablePassthrough(), metadata=None), 'generate_initial_summary': Node(id='generate_initial_summary', name='generate_initial_summary', data=generate_initial_summary(tags=None, recurse=True, explode_args=False, func_accepts_config=False, func_accepts={}), metadata=None), 'generate_refined_summary': Node(id='generate_refined_summary', name='generate_refined_summary', data=generate_refined_summary(tags=None, recurse=True, explode_args=False, func_accepts_config=False, func_accepts={}), metadata=None), '__end__': Node(id='__end__', name='__end__', data=None, metadata=None)}, edges=[Edge(source='__start__', target='generate_initial_summary', data=None, conditional=False), Edge(source='generate_initial_summary', target='__end__', data=None, conditional=False)])

INFERENCE

In [18]:
async for step in graph.astream(
    {"content": [doc.page_content for doc in documents]},
    stream_mode="values",
):
    if summary := step.get("summary"): ## if the new summary is same as previous summary
        print(summary)

content='"An Introduction to Transformers" by Richard E. Turner provides a mathematically precise and intuitive overview of the transformer architecture, a neural network component crucial for advances in natural language processing, computer vision, and spatio-temporal modeling. Unlike many introductions, this note offers detailed mathematical descriptions and insights into design choices, assuming familiarity with basic machine learning concepts. The text does not cover training methods, focusing instead on the architecture itself.' additional_kwargs={} response_metadata={'token_usage': {'prompt_tokens': 361, 'total_tokens': 456, 'completion_tokens': 95}, 'model_name': 'mistral-large-latest', 'model': 'mistral-large-latest', 'finish_reason': 'stop'} id='run--63af2645-548f-49d8-8bee-6ff2118078fe-0' usage_metadata={'input_tokens': 361, 'output_tokens': 95, 'total_tokens': 456}
content='### Refined Summary\n\n"An Introduction to Transformers" by Richard E. Turner offers a comprehensive 

In [19]:
summary.pretty_print()

================================== Ai Message ==================================

### Refined Summary

"An Introduction to Transformers" by Richard E. Turner offers a thorough and mathematically detailed examination of the transformer architecture, which is essential in neural networks for natural language processing, computer vision, and spatio-temporal modeling. The note assumes a basic understanding of machine learning concepts and focuses on the architecture rather than training methods.

### Key Points:

1. **Input Data Format**:
   - Transformers process data as sets or sequences of N tokens, each of dimension D.
   - These tokens can be represented in a D×N matrix \( X^{(0)} \).
   - Examples include splitting text into words or sub-words and dividing images into patches, each represented as vectors.

2. **Goal**:
   - The transformer processes the input data \( X^{(0)} \) to produce a representation matrix \( X^{(M)} \), also of size D×N.
   - This representation can be used fo